In [1]:
#!/usr/bin/env python
# coding: utf-8

# Standard library imports
import itertools
from itertools import product
#import logging
import multiprocessing
import os
import shutil
import sys
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Iterable, Tuple

# Third-party imports
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.widgets  # Cursor
import numpy as np
import pandas as pd
import polars as pl
import scipy.integrate
import scipy.ndimage.interpolation
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import mean_squared_error

### Load Colorado 

In [17]:
benchmark_schema = {
    'fips': pl.Int64,
    'days_from_start': pl.Int64,
    'intercept_TLGRF': pl.Float64,
    'r_TLGRF': pl.Float64,
    'county': pl.Utf8,  # Equivalent to String in Polars
    'state': pl.Utf8,   # Equivalent to String in Polars
    'date': pl.Date,
    'rolled_cases': pl.Float64,
    'log_rolled_cases': pl.Float64,
    'shifted_log_rolled_cases': pl.Float64,
    'TLGRF_predicted_log_rolled_cases': pl.Float64
}

benchmark_TLGRF_dataset = pl.read_csv("benchmark_TLGRF_dataset.csv", schema=benchmark_schema)
benchmark_colorado_dataset = benchmark_TLGRF_dataset.filter(
        (pl.col("state") == "Colorado") & 
        (pl.col("TLGRF_predicted_log_rolled_cases").is_not_null()) & 
        (pl.col("shifted_log_rolled_cases").is_not_null())
    ).sort(["fips","date"])

In [19]:
fips_list = benchmark_colorado_dataset.select("fips").unique()
fips_list

fips
i64
8001
8003
8005
8007
8009
…
8117
8119
8121


### Load Best Windows for Each Cluster

In [10]:
tcv_best_windows = pl.read_csv("../benchmark_tcv_kmeans_code/one_point_validation/merged_kmeans_tcv_validation/merged_K=1.csv")
tcv_best_windows = tcv_best_windows.with_columns(pl.col("date").str.strptime(pl.Date, "%Y-%m-%d"))

In [24]:
tcv_best_windows.schema

Schema([('K', Float64),
        ('k', Float64),
        ('date', Date),
        ('best_mae_window', Float64),
        ('best_rmse_window', Float64)])

In [11]:
tcv_best_windows

K,k,date,best_mae_window,best_rmse_window
f64,f64,date,f64,f64
1.0,0.0,2020-01-28,2.0,2.0
1.0,0.0,2020-01-29,2.0,2.0
1.0,0.0,2020-01-30,2.0,2.0
1.0,0.0,2020-01-31,3.0,3.0
1.0,0.0,2020-02-01,5.0,5.0
…,…,…,…,…
1.0,0.0,2022-12-27,2.0,2.0
1.0,0.0,2022-12-28,2.0,2.0
1.0,0.0,2022-12-29,2.0,2.0


### Load `beta_wsize`

In [21]:
Fixed_windows_all_beta = pl.read_csv("../benchmark_fixed_window/Fixed_windows_all_beta.csv")
Fixed_windows_all_beta = Fixed_windows_all_beta.with_columns(pl.col("date").str.strptime(pl.Date, "%Y-%m-%d"))
Fixed_windows_all_beta_Colorado = Fixed_windows_all_beta.filter(pl.col("fips").is_in(fips_list))

In [25]:
Fixed_windows_all_beta_Colorado.schema

Schema([('fips', Int64),
        ('days_from_start', Int64),
        ('date', Date),
        ('log_rolled_cases', Float64),
        ('shifted_log_rolled_cases', Float64),
        ('beta_wsize=2', Float64),
        ('beta_wsize=3', Float64),
        ('beta_wsize=4', Float64),
        ('beta_wsize=5', Float64),
        ('beta_wsize=6', Float64),
        ('beta_wsize=7', Float64),
        ('beta_wsize=8', Float64),
        ('beta_wsize=9', Float64),
        ('beta_wsize=10', Float64),
        ('beta_wsize=11', Float64),
        ('beta_wsize=12', Float64),
        ('beta_wsize=13', Float64),
        ('beta_wsize=14', Float64)])

In [23]:
Fixed_windows_all_beta_Colorado

fips,days_from_start,date,log_rolled_cases,shifted_log_rolled_cases,beta_wsize=2,beta_wsize=3,beta_wsize=4,beta_wsize=5,beta_wsize=6,beta_wsize=7,beta_wsize=8,beta_wsize=9,beta_wsize=10,beta_wsize=11,beta_wsize=12,beta_wsize=13,beta_wsize=14
i64,i64,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
8001,57,2020-03-18,2.018705,3.075995,null,null,null,null,null,null,null,null,null,null,null,null,null
8001,58,2020-03-19,2.160034,3.310022,0.141328,null,null,null,null,null,null,null,null,null,null,null,null
8001,59,2020-03-20,2.29829,3.578347,0.138257,0.139792,null,null,null,null,null,null,null,null,null,null,null
8001,60,2020-03-21,2.406945,3.858321,0.108655,0.123456,0.130298,null,null,null,null,null,null,null,null,null,null
8001,61,2020-03-22,2.550561,4.103116,0.143616,0.126135,0.128024,0.131062,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
8125,1071,2022-12-27,3.916868,3.867324,0.017207,0.030783,0.033994,0.035514,0.037549,0.038357,0.035497,0.033474,0.03042,0.028035,0.026152,0.024523,0.023015
8125,1072,2022-12-28,3.953165,3.830813,0.036297,0.026752,0.03108,0.033122,0.034456,0.036247,0.037165,0.035115,0.033529,0.030975,0.028872,0.027138,0.025595
8125,1073,2022-12-29,3.955903,3.792918,0.002738,0.019517,0.020502,0.025471,0.028532,0.03068,0.032912,0.034313,0.033155,0.03216,0.030209,0.028516,0.027061


### Select the best beta for each entry

In [33]:
# Create a new column for the dynamic beta column name
tcv_best_windows = tcv_best_windows.with_columns(
    (pl.lit("beta_wsize=") + pl.col("best_rmse_window").cast(int).cast(str)).alias("best_beta_col")
)

# Melt the Fixed_windows_all_beta_Colorado to have a single `beta_values` column
melted_fixed_windows = Fixed_windows_all_beta_Colorado.melt(
    id_vars=["fips", "date"], 
    value_vars=[col for col in Fixed_windows_all_beta_Colorado.columns if col.startswith("beta_wsize=")],
    variable_name="beta_name", 
    value_name="beta_values"
)

# Join on `date` and match `best_beta_col` to `beta_name`
result_df = (
    melted_fixed_windows.join(
        tcv_best_windows, 
        on="date", 
        how="inner"
    )
    .filter(pl.col("beta_name") == pl.col("best_beta_col"))
    .select(["fips", "date", "beta_values"])
    .rename({"beta_values": "r_tcv"})
).with_columns(
        pl.col("r_tcv")
        .fill_null(strategy="backward")  # Backfill initial NaNs
        .fill_null(strategy="forward")  # Forward fill remaining NaNs
    )

result_df = result_df.join(Fixed_windows_all_beta_Colorado.select(["date","days_from_start"]), on=["date"])
result_df = result_df.select(["fips","days_from_start","date","r_tcv"]).sort(["fips","days_from_start"])

print(result_df)


/tmp/ipykernel_2643848/2523865695.py:7: DeprecationWarning: `DataFrame.melt` is deprecated. Use `unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`
  melted_fixed_windows = Fixed_windows_all_beta_Colorado.melt(


shape: (4_034_251, 4)
┌──────┬─────────────────┬────────────┬───────────┐
│ fips ┆ days_from_start ┆ date       ┆ r_tcv     │
│ ---  ┆ ---             ┆ ---        ┆ ---       │
│ i64  ┆ i64             ┆ date       ┆ f64       │
╞══════╪═════════════════╪════════════╪═══════════╡
│ 8001 ┆ 57              ┆ 2020-03-18 ┆ 0.108655  │
│ 8001 ┆ 57              ┆ 2020-03-18 ┆ 0.108655  │
│ 8001 ┆ 57              ┆ 2020-03-18 ┆ 0.108655  │
│ 8001 ┆ 57              ┆ 2020-03-18 ┆ 0.108655  │
│ 8001 ┆ 57              ┆ 2020-03-18 ┆ 0.108655  │
│ …    ┆ …               ┆ …          ┆ …         │
│ 8125 ┆ 1075            ┆ 2022-12-31 ┆ -0.016774 │
│ 8125 ┆ 1075            ┆ 2022-12-31 ┆ -0.016774 │
│ 8125 ┆ 1075            ┆ 2022-12-31 ┆ -0.016774 │
│ 8125 ┆ 1075            ┆ 2022-12-31 ┆ -0.016774 │
│ 8125 ┆ 1075            ┆ 2022-12-31 ┆ -0.016774 │
└──────┴─────────────────┴────────────┴───────────┘


In [35]:
result_df.write_csv("tcv_beta.csv")

In [36]:
result_df.write_parquet("tcv_beta.parquet")